In [1]:
import torchref

/tmp/ipykernel_43954/144460258.py:1: UserWarning: TorchRef auto-configured 4 threads. Set TORCHREF_NUM_THREADS to override.
  import torchref


In [2]:
'''
Basic structure loading
'''

pdb_file = '/das/work/p17/p17490/Peter/Library/torchref/example_notebooks/1DAW.pdb'

from torchref.model import ModelFT

instance_model_ft = ModelFT().load_pdb(pdb_file)

from torchref.model import Model

instance_model = Model().load_pdb(pdb_file)


Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (360, 144, 108)
  ✓ Using direct indexing (no interpolation)
Loaded 3051 atoms


In [3]:
'''
Basic data handling
'''

from torchref.io import ReflectionData

mtz = '/das/work/p17/p17490/Peter/Library/torchref/example_notebooks/1DAW.mtz'

reflection_data = ReflectionData().load_mtz(mtz)


FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged


In [4]:
'''
Basic scaling
'''

from torchref.scaling import Scaler

scaling_instance = Scaler(model=instance_model_ft, data=reflection_data)

scaling_instance.initialize()

print(scaling_instance.rfactor())

scaling_instance.refine_lbfgs()

print(scaling_instance.rfactor())




Initialized Scaler with 20 bins.
Calculating initial scale factors using 20 bins.


/das/work/p17/p17490/CONDA/torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


(0.2883461117744446, 0.33538785576820374)
Refining scales with LBFGS...
Scale refinement complete. rwork: 0.2100, rfree: 0.2740

Final Scale Parameters: 
  log_scale: tensor([-6.2381, -6.1446, -6.1089, -6.0416, -6.0229, -5.9816, -5.9337, -5.8897,
        -5.8758, -5.8257, -5.7901, -5.7363, -5.6681, -5.6116, -5.5564, -5.4739,
        -5.3798, -5.3505, -5.3462, -5.3506])
  U: tensor([-0.3220, -0.2307, -0.1451, -0.0031, -0.1590, -0.0045])
  solvent.log_k_solvent: -0.9721982479095459
  solvent.b_solvent: 30.828311920166016
  solvent.phase_offset: -0.0013634959468618035
(0.20997777581214905, 0.27399057149887085)


/das/work/p17/p17490/CONDA/torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


In [5]:
#Setup refinement object

from torchref.refinement import LBFGSRefinement

refinement_object = LBFGSRefinement(pdb=pdb_file, data_file=mtz)

print(refinement_object.get_rfactor()) # initial rfactor


FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (160, 72, 54)
  ✓ Using direct indexing (no interpolation)
Initialized Scaler with 10 bins.
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
------------------------------------------------

/das/work/p17/p17490/CONDA/torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


In [6]:
#Shake coords

refinement_object.model.shake_coords(0.1) # shake coordinates by 0.1 Angstroms

print(refinement_object.get_rfactor()) # rfactor after shaking


(0.2340780347585678, 0.28340667486190796)


In [7]:
#Create the loss state for refinement and add target info, meta info, and weights

loss_state = refinement_object.create_loss_state()

refinement_object.add_target_info_to_state(loss_state) # add target info to the loss state
refinement_object.populate_state_meta(loss_state) # populate the state with meta info

refinement_object.update_weights(loss_state) # update/set weights in the loss state

print(loss_state.weights) # weights
print("Total loss", loss_state.aggregate())


{'xray': 0.9574738144874573, 'geometry/bond': 71.48390197753906, 'geometry/angle': 29.239818572998047, 'geometry/torsion': 3.6619229316711426, 'geometry/planarity': 7.9569010734558105, 'geometry/chiral': 35.18427658081055, 'geometry/nonbonded': 1.4662721157073975, 'adp/simu': 1.0914393663406372, 'adp/locality': 4.187494277954102, 'adp/KL': 2.266031265258789, 'geometry': 1.1340019702911377, 'adp': 1.1340019702911377}
Total loss tensor(6213.4111, grad_fn=<AddBackward0>)


In [ ]:

#Run a refinement step

parameters = refinement_object.parameters() 

refinement_object._optimize_lbfgs(loss_state, parameters, max_iter=100,nsteps=1) # run 100 iterations of L-BFGS and step once

print(refinement_object.get_rfactor()) # final rfactor


In [ ]:
refinement_object.cuda() # move to GPU
loss_state.cuda()


Model moved to device: cuda


LBFGSRefinement(
  (model): ModelFT(
    (_cache): TensorDict({ae18b024e04219e76e99f0266f0e2da9baecb4a2: tensor([ -46.1953+15.3784j,   17.5593+221.1547j,   56.8098-117.9367j,
             ...,   99.7449-139.8228j,   -0.8217+40.3482j,
            -126.1186+23.5122j], device='cuda:0'), fd70bad0ce3ff2587d6fe93902e5c239dfc228fb: tensor([  23.0069+40.7331j,  -25.5959+206.2394j,   49.4721-81.7420j,
             ...,   93.6422-124.9881j,    6.0892+42.8779j,
            -104.5171+32.0344j], device='cuda:0'), 3801e375dd9d4829451eb41d9b2e8ef598a78720: tensor([  22.9624+40.6542j,  -25.5229+206.2487j,   49.5219-81.7650j,
             ...,   93.5813-124.9601j,    5.9891+42.7958j,
            -104.6257+32.0419j], device='cuda:0'), e2979b5cf2bc9389dd9d6c3387af4b3b35dc695a: tensor([  22.5624+39.9510j,  -24.8473+206.3272j,   49.9716-81.9810j,
             ...,   93.0286-124.7071j,    5.0808+42.0632j,
            -105.5981+32.1096j], device='cuda:0'), ff8419199b9c655b1d8e3e71ba48183c4d442d27: tensor([  

In [ ]:
#Run a refinement step

parameters = refinement_object.parameters() 

refinement_object._optimize_lbfgs(loss_state, parameters, max_iter=100,nsteps=1) # run 100 iterations of L-BFGS and step once

print(refinement_object.get_rfactor()) # final rfactor

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!